# Paper Figures — Tier B

Population and secondary main-text figures for the MALCA dipper paper.

Outputs go to `output/notebooks/paper_figures/`.

Wavelet power spectra are **not** duplicated here — use `july1_dipper_wavelet_atlas.ipynb` for the full per-candidate WWZ atlas.

## Parameters

In [ ]:
%matplotlib inline

from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from malca.io.notebook_paths import find_repo_root

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from malca.plotting.notebook_display import reload_plotting_modules, show_figure

reload_plotting_modules()

from malca.plotting.dipper_wavelet import generate_wavelet_atlas, read_review_labeled_candidates
from malca.plotting.lightcurve_publication import apply_publication_rcparams
from malca.plotting.paper_figures import (
    PaperFigureContext,
    generate_tier_b,
    load_dippers,
    plot_age_histogram,
    plot_dip_depth_histogram,
    plot_dip_duration_histogram,
    plot_halpha_vs_m,
    plot_ks_w3_w4_observed,
    plot_mollweide_sky_sfr,
    plot_period_histogram,
    plot_sed_alpha_vs_halpha,
    plot_sed_alpha_vs_ir_rms,
)

RUN_NAME = "dat3-full-extended_2026-07-01-v4"
RUN_ROOT = REPO_ROOT / "output" / "runs" / RUN_NAME
REVIEW_DB = RUN_ROOT / "review" / "review.db"
OUTPUT_DIR = REPO_ROOT / "output" / "notebooks" / "paper_figures"
WAVELET_OUTPUT_DIR = REPO_ROOT / "output" / "notebooks" / "paper_figures" / "wavelet_atlas"

EXPORT_PDF = True
EXPORT_PNG = True
RUN_WAVELET_ATLAS = False  # set True to regenerate wavelet figures (slow)
WAVELET_MAX_CANDIDATES = None  # int for smoke test

ctx = PaperFigureContext(
    repo_root=REPO_ROOT,
    run_root=RUN_ROOT,
    review_db=REVIEW_DB,
    output_dir=OUTPUT_DIR,
    export_pdf=EXPORT_PDF,
    export_png=EXPORT_PNG,
    show_inline=True,
)
ctx.output_dir.mkdir(parents=True, exist_ok=True)

apply_publication_rcparams(plt)
print(f"Review DB: {REVIEW_DB}")
print(f"Output dir: {OUTPUT_DIR}")

## Load dippers

In [ ]:
dippers = load_dippers(REVIEW_DB)
display(Markdown(f"Reviewed dippers: **{len(dippers):,}**"))

DatabaseError: Execution failed on sql '
        SELECT
            c.candidate_id,
            c.stats_amplitude,
            c.stats_photometry_median_mag,
            c.stats_error_and_snr_stats_error_median,
            c.stats_variability_quasi_periodicity_q,
            c.stats_variability_flux_asymmetry_m,
            c.stats_variability_periodic_feature_period_source,
            c.ra,
            c.dec,
            c.gal_l,
            c.gal_b,
            c.bp_rp,
            c.bp_rp_err,
            c.phot_g_mean_mag,
            c.phot_g_mean_mag_err,
            c.A_v_3d,
            c.A_v_3d_err,
            c.age50,
            c.period_consensus_days,
            c.period_primary_source,
            c.tmass_k, c.tmass_k_err,
            c.w1, c.w1_err, c.w2, c.w2_err, c.w3, c.w3_err, c.w4, c.w4_err,
            c.sed_alpha,
            c.iphas_r_ha, c.vphas_r_ha,
            c.pmra, c.pmdec, c.parallax,
            c.distance_gspphot,
            c.dip_best_mag_event,
 

## B1 — $K_s-W3$ vs. $K_s-W4$ (observed disk color–color)

In [ ]:
show_figure(*plot_ks_w3_w4_observed(ctx, dippers))

## B2 — SED slope $\alpha$ vs. H$\alpha$ (using $r-H\alpha$ color as EW proxy)

In [ ]:
show_figure(*plot_sed_alpha_vs_halpha(ctx, dippers))

## B3 — Period histogram

In [ ]:
show_figure(*plot_period_histogram(ctx, dippers))

## B4 — H$\alpha$ vs. $M$

In [ ]:
show_figure(*plot_halpha_vs_m(ctx, dippers))

## B5 — SED $\alpha$ vs. IR RMS (AllWISE W1 intrinsic scatter)

In [ ]:
show_figure(*plot_sed_alpha_vs_ir_rms(ctx, dippers))

## B6 — Mollweide sky map with SFR footprints and PM vectors

In [ ]:
show_figure(*plot_mollweide_sky_sfr(ctx, dippers))

## B7 — Age histogram

In [ ]:
show_figure(*plot_age_histogram(ctx, dippers))

## B8 — Wavelet power spectra

Delegates to the existing wavelet atlas exporter. For interactive exploration, use `july1_dipper_wavelet_atlas.ipynb` directly.

In [ ]:
if RUN_WAVELET_ATLAS:
    wavelet_candidates = read_review_labeled_candidates(REVIEW_DB, event_class="dipper")
    WAVELET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    manifest = generate_wavelet_atlas(
        wavelet_candidates,
        run_root=RUN_ROOT,
        output_dir=WAVELET_OUTPUT_DIR,
        max_candidates=WAVELET_MAX_CANDIDATES,
        write_png=True,
        write_pdf=True,
        build_atlas_pdf=True,
    )
    display(manifest.head())
else:
    print("Set RUN_WAVELET_ATLAS=True to export wavelet figures.")

## B9 — Dip-depth histogram

In [ ]:
show_figure(*plot_dip_depth_histogram(ctx, dippers))

## B10 — Dip-duration histogram

In [ ]:
show_figure(*plot_dip_duration_histogram(ctx, dippers))

## Batch export (all Tier B figures except wavelets)

In [ ]:
outputs = generate_tier_b(ctx)
for name, (paths, _fig) in outputs.items():
    print(name, "->", ", ".join(str(p) for p in paths))